**Installing  libraries**

In [ ]:
!pip install pandas numpy geopandas matplotlib seaborn rioxarray rasterio

**Importing libraries**

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import rioxarray
from rasterio.plot import plotting_extent
import rasterio
import matplotlib.patches as mpatches 

**Create folders in Colab**

In [ ]:
BASE_DIR = Path(".").resolve()

# Set paths to subfolders
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

# Create output folder automatically if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project initialized inside: {BASE_DIR.name}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

**Loading dataset**

In [ ]:
csv_path = DATA_DIR / "Data_Chtouka_Reduced.csv"
df = pd.read_csv(csv_path)


**Convert date from object type to datetime for proper date handling**

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [ ]:
df['year'] = df['date'].dt.year

**Convert to a GeoDataFrame**

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:26192"
)

**Transform to WGS84 (EPSG:4326)**

In [ ]:
gdf = gdf.to_crs("EPSG:4326")

**Save the processed data as a CSV file in the output folder**

In [ ]:
# Create the output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define the file path using pathlib
save_path = OUTPUT_DIR / "Processed_qgis_Data.csv"

# Save the file (index=False prevents adding an extra row number column)
gdf.to_csv(save_path, index=False)
print(f"File successfully saved at: {save_path}")

**Reading limite shape file**

In [ ]:
path_limite = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "Limite" / "00_LimiteChtoukaWGS.shp"
limite = gpd.read_file(path_limite)

**Groundwater Wells over Selected Layer — Configurable Shapefile Plot**

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# CHANGE THIS ONE LINE to plot a different shapefile.
# Copy one of the paths below and paste it in as shapefile_path.
# ============================================================
#
# path_geologie        = DATA_DIR / "GeologyBuffer_new" / "Geology.shp"                                                     
# path_soil            = DATA_DIR / "SoilType_latest_one" / "SoilTypesChtoukaBuffer10km_f.shp"                             
# path_rivers          = DATA_DIR / "hydrorivers" / "Chtouka_Hydrorivers_Clipped.shp"                                      - 
# path_farms           = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "FermesIrriguéesDessalemen" / "04_FermesIrriguéesDessalemen.shp"    
# path_wells_bareholes = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "Wells_Bareholes" / "05_Wells and Bareholes.shp"           
# path_perimeters      = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "Irrigated_perimeters_Fig6&7" / "06_Irrigated_perimeters_Fig6&7.shp"   
# path_desalinization  = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "Desalinization_Network" / "08_Reseau_dessalement.shp"      
#
shapefile_path =  DATA_DIR / "GeologyBuffer_new" / "Geology.shp"    

# ------------------------------------------------------------
# Column Lookup
# Only shapefiles that have a meaningful category column need
# an entry here. Anything not listed just plots as a plain
# outline/shape (see comments above).
# ------------------------------------------------------------
column_lookup = {
    "Geology.shp": "desc_",
    "SoilTypesChtoukaBuffer10km_f.shp": "TEXTURE",
    "Chtouka_Hydrorivers_Clipped.shp": "ORD_CLAS"
}
color_column = column_lookup.get(shapefile_path.name)

# --- Load & Reproject shapefiles ---
layer = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# Ensure 'limite' is reprojected as well (assuming 'limite' exists in your environment)
if 'limite' in locals():
    limite = limite.to_crs("EPSG:4326")

# --- Set up the plot ---
fig, ax = plt.subplots(figsize=(12, 10))

# 1. Draw boundary first
if 'limite' in locals():
    limite.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=1.5, zorder=1)

# 2. Draw the chosen shapefile
if color_column and color_column in layer.columns:
    # Explicitly set categorical=True so the column renders a discrete legend
    layer.plot(
        ax=ax,
        column=color_column,
        categorical=True,
        cmap='tab20b',
        alpha=0.85,
        legend=True,
        legend_kwds={
            'title': f'{shapefile_path.stem} ({color_column})',
            'loc': 'upper left',
            'bbox_to_anchor': (1.02, 1),
            'frameon': True
        },
        zorder=2
    )
else:
    layer.plot(ax=ax, color='blue', linewidth=1, alpha=0.7, zorder=2)

# 3. Draw wells on top
gdf = gdf.to_crs("EPSG:4326")
gdf.plot(
    ax=ax,
    color='red',
    markersize=30,
    edgecolor='white',
    zorder=3
)

# --- Merge Groundwater Wells marker into the legend ---
leg = ax.get_legend()
if leg:
    # Create custom proxy handle for the wells point
    well_handle = Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=f'Groundwater Wells ({len(gdf)})',
        markerfacecolor='red',
        markeredgecolor='white',
        markersize=8
    )

    # Append the well entry to existing legend handles & labels
    handles = leg.legend_handles + [well_handle]
    labels = [t.get_text() for t in leg.get_texts()] + [f'Groundwater Wells ({len(gdf)})']

    # Re-draw the combined legend
    ax.legend(
        handles=handles,
        labels=labels,
        loc='upper left',
        bbox_to_anchor=(1.02, 1),
        title="Legend",
        frameon=True
    )

# --- Labels & Layout ---
ax.set_title(f"Wells over {shapefile_path.stem}", fontsize=14, pad=10)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Reserve space on the right for the legend
plt.subplots_adjust(right=0.78)
plt.show()

**Groundwater Wells over Selected Raster — Configurable Raster (.tif) Plot**

In [ ]:
import rasterio
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ============================================================
# PATH SELECTION
# Change active path by commenting/uncommenting:
# ============================================================

# path_landcover2017 = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "LULC_10m_2024_2017" / "Landcover2017" / "Landcover2017.tif"   -> categorical (land cover classes)
# path_demf          = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "demf" / "01_demfChtouka.tif"                                  -> continuous (elevation, needs scale/offset)
raster_path = DATA_DIR / "DataBaseChtoukaFinal_WGS" / "demf" / "01_demfChtouka.tif"  


# ------------------------------------------------------------
# Metadata Config (Used when Land Cover is active)
# ------------------------------------------------------------
land_cover_meta = {
    1: ("Water", "blue"),
    2: ("Trees", "green"),
    5: ("Crops", "orange"),
    7: ("Built Area", "grey"),
    8: ("Bare Ground", "yellow"),
    11: ("Rangeland", "lightgreen")
}

# ------------------------------------------------------------
# STEP 1: Prepare Wells
# ------------------------------------------------------------
wells_unique = gdf.drop_duplicates(subset=[gdf.geometry.name]).copy()

# ------------------------------------------------------------
# STEP 2: Open Raster Data & Apply Scale/Offset
# ------------------------------------------------------------
with rasterio.open(raster_path) as src:
    data = src.read(1, masked=True)
    raster_extent = plotting_extent(src)
    raster_crs = src.crs

    scale = src.scales[0] if src.scales and src.scales[0] is not None else 1.0
    offset = src.offsets[0] if src.offsets and src.offsets[0] is not None else 0.0
    data_scaled = data * scale + offset

# ------------------------------------------------------------
# STEP 3: Align Spatial Projections
# ------------------------------------------------------------
wells_raster_crs = wells_unique.to_crs(raster_crs)

# ------------------------------------------------------------
# STEP 4: Determine Dynamic Well Colors Based on Raster Type
# ------------------------------------------------------------
is_landcover = "landcover" in raster_path.name.lower()

# Cyan for Land Cover (avoids pink/red overlap), Red for DEM
well_color = "cyan" if is_landcover else "red"
well_edge = "black" if is_landcover else "white"

# ------------------------------------------------------------
# STEP 5: Render Map
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 15))

# Dynamic well marker graphic for legend
well_legend_handle = Line2D(
    [0], [0], marker='o', color='w', markerfacecolor=well_color,
    markeredgecolor=well_edge, markeredgewidth=1.2, markersize=10, label='Study Wells'
)

if is_landcover:
    # --- Categorical Display (Land Cover) ---
    img = ax.imshow(data_scaled, cmap="tab20", origin="upper", extent=raster_extent)

    # Build discrete land cover legend
    legend_handles = [
        mpatches.Patch(color=color, label=f"{label} (ID {val})")
        for val, (label, color) in land_cover_meta.items()
    ]
    legend_handles.append(well_legend_handle)

    ax.legend(
        handles=legend_handles, 
        loc='lower right', 
        title="Land Cover Classes",
        fontsize=10, 
        frameon=True, 
        facecolor='white'
    )

else:
    # --- Continuous Display (DEM / Elevation) ---
    img = ax.imshow(data_scaled, cmap="viridis", origin="upper", extent=raster_extent)
    
    # Continuous scale bar
    cbar = plt.colorbar(img, ax=ax, shrink=0.7, pad=0.03)
    cbar.set_label("Elevation (meters above sea level)", fontsize=11)

    ax.legend(
        handles=[well_legend_handle], 
        loc='lower right', 
        fontsize=10, 
        frameon=True, 
        facecolor='white'
    )

# Overlay study wells with adaptive colors
wells_raster_crs.plot(
    ax=ax, 
    color=well_color, 
    edgecolor=well_edge, 
    linewidth=1.2, 
    markersize=45, 
    zorder=5
)

# Lock plot domain to exact raster extent
ax.set_xlim(raster_extent[0], raster_extent[1])
ax.set_ylim(raster_extent[2], raster_extent[3])

# Grid & Axis Labels
ax.set_title(f"Wells over {raster_path.stem}", fontsize=14, fontweight='bold')
ax.set_xlabel("Easting / Longitude", fontsize=11)
ax.set_ylabel("Northing / Latitude", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()